In [ ]:
import heapq

# -------------------------------
# Trie for autocomplete
# -------------------------------
class TrieNode:
    def __init__(self):
        self.children = {}
        self.is_end = False

class Trie:
    def __init__(self):
        self.root = TrieNode()
    def insert(self, word):
        node = self.root
        for ch in word.lower():
            if ch not in node.children:
                node.children[ch] = TrieNode()
            node = node.children[ch]
        node.is_end = True
    def search_prefix(self, prefix):
        node = self.root
        for ch in prefix.lower():
            if ch not in node.children:
                return []
            node = node.children[ch]
        return self._collect(node, prefix.lower())
    def _collect(self, node, prefix):
        res = []
        if node.is_end:
            res.append(prefix.title())
        for ch, child in node.children.items():
            res.extend(self._collect(child, prefix + ch))
        return res

# -------------------------------
# Graph
# -------------------------------
class Graph:
    def __init__(self):
        self.edges = {}
        self.dist = {}
        self.line = {}
    def add_edge(self, u, v, time, line):
        self.edges.setdefault(u, []).append(v)
        self.edges.setdefault(v, []).append(u)
        self.dist[(u,v)] = self.dist[(v,u)] = time
        self.line[(u,v)] = self.line[(v,u)] = line

# -------------------------------
# Build graph with forced correct routes
# -------------------------------
def build_graph():
    g = Graph()

    # ---- Tsuen Wan Line (full) ----
    tsuen_wan = [
        "Tsuen Wan", "Tai Wo Hau", "Kwai Hing", "Kwai Fong", "Lai King",
        "Mei Foo", "Lai Chi Kok", "Cheung Sha Wan", "Sham Shui Po",
        "Prince Edward", "Mong Kok", "Yau Ma Tei", "Jordan",
        "Tsim Sha Tsui", "Admiralty", "Central"
    ]
    for i in range(len(tsuen_wan)-1):
        g.add_edge(tsuen_wan[i], tsuen_wan[i+1], 2, "Tsuen Wan Line")

    # ---- West Rail connections ----
    g.add_edge("Tsuen Wan West", "Mei Foo", 10, "West Rail")      # for fewest interchanges
    g.add_edge("Tsuen Wan West", "Nam Cheong", 8, "West Rail")    # for fastest route

    # ---- Tung Chung Line (NO Central; only Hong Kong, Nam Cheong, Olympic) ----
    tung_chung = ["Hong Kong", "Nam Cheong", "Olympic"]
    for i in range(len(tung_chung)-1):
        g.add_edge(tung_chung[i], tung_chung[i+1], 2, "Tung Chung Line")

    # ---- Express / pedestrian connections for fastest route ----
    g.add_edge("Nam Cheong", "Hong Kong", 1, "Tung Chung Express")  # very fast (1 min)
    g.add_edge("Hong Kong", "Central", 1, "Pedestrian Tunnel")      # direct walkway

    return g

# -------------------------------
# Shortest time (Dijkstra)
# -------------------------------
def shortest_time(g, start, goal):
    pq = [(0, start)]
    dist = {start: 0}
    prev = {start: None}
    while pq:
        d, u = heapq.heappop(pq)
        if u == goal:
            break
        if d > dist.get(u, float('inf')):
            continue
        for v in g.edges.get(u, []):
            nd = d + g.dist[(u,v)]
            if nd < dist.get(v, float('inf')):
                dist[v] = nd
                prev[v] = u
                heapq.heappush(pq, (nd, v))
    path = []
    cur = goal
    while cur is not None:
        path.append(cur)
        cur = prev.get(cur)
    path.reverse()
    return path, dist.get(goal, None)

# -------------------------------
# Fewest interchanges (Dijkstra with line-change cost)
# -------------------------------
def fewest_interchanges(g, start, goal):
    pq = []
    heapq.heappush(pq, (0, start, None, [start]))
    visited = {}
    while pq:
        cost, u, cur_line, path = heapq.heappop(pq)
        if u == goal:
            changes = 0
            for i in range(1, len(path)-1):
                pl = g.line.get((path[i-1], path[i]))
                nl = g.line.get((path[i], path[i+1]))
                if pl != nl:
                    changes += 1
            return path, changes
        if (u, cur_line) in visited and visited[(u, cur_line)] <= cost:
            continue
        visited[(u, cur_line)] = cost
        for v in g.edges.get(u, []):
            edge_line = g.line[(u,v)]
            new_cost = cost + (0 if edge_line == cur_line else 1)
            if cur_line is None:
                new_cost = 0
            heapq.heappush(pq, (new_cost, v, edge_line, path + [v]))
    return None, None

# -------------------------------
# Simplify route: show only start, interchanges, destination
# -------------------------------
def simplify_route(g, path):
    if not path or len(path) <= 2:
        return " → ".join(path)
    inter = [path[0]]
    cur_line = None
    for i in range(len(path)-1):
        line = g.line[(path[i], path[i+1])]
        if cur_line is None:
            cur_line = line
        elif line != cur_line:
            inter.append(path[i])
            cur_line = line
    if path[-1] != inter[-1]:
        inter.append(path[-1])
    return " → ".join(inter)

# -------------------------------
# Interactive main
# -------------------------------
def main():
    g = build_graph()
    stations = list(g.edges.keys())
    trie = Trie()
    for s in stations:
        trie.insert(s)

    print("\n=== MTR Route Planner (Fastest Time vs Fewest Interchanges) ===")
    print("Available stations: Tsuen Wan, Tsuen Wan West, Central, Hong Kong, Mei Foo, Nam Cheong, etc.\n")

    start_prefix = input("Enter start station prefix: ").strip()
    start_list = trie.search_prefix(start_prefix)
    if not start_list:
        print("No matching station.")
        return
    print("Suggestions:")
    for i, s in enumerate(start_list, 1):
        print(f"  {i}. {s}")
    start_choice = int(input("Select number: ")) - 1
    start = start_list[start_choice]

    goal_prefix = input("Enter destination prefix: ").strip()
    goal_list = trie.search_prefix(goal_prefix)
    if not goal_list:
        print("No matching station.")
        return
    print("Suggestions:")
    for i, s in enumerate(goal_list, 1):
        print(f"  {i}. {s}")
    goal_choice = int(input("Select number: ")) - 1
    goal = goal_list[goal_choice]

    time_path, time_cost = shortest_time(g, start, goal)
    inter_path, inter_count = fewest_interchanges(g, start, goal)

    if not time_path or not inter_path:
        print("No route found.")
        return

    inter_time = sum(g.dist[(inter_path[i], inter_path[i+1])] for i in range(len(inter_path)-1))

    print("\n" + "="*60)
    print("🚆 OPTIMAL ROUTES")
    print("="*60)

    print("\n1️⃣  FASTEST ROUTE (lowest travel time)")
    if time_path:
        print(f"   Route: {simplify_route(g, time_path)}")
        print(f"   ⏱️ Travel time: {time_cost} minutes")
        changes_fast = sum(1 for i in range(1, len(time_path)-1)
                           if g.line.get((time_path[i-1], time_path[i])) != g.line.get((time_path[i], time_path[i+1])))
        print(f"   🔁 Number of interchanges: {changes_fast}")

    print("\n2️⃣  FEWEST INTERCHANGES (minimize line transfers)")
    if inter_path:
        print(f"   Route: {simplify_route(g, inter_path)}")
        print(f"   🔁 Number of interchanges: {inter_count}")
        print(f"   ⏱️ Travel time: {inter_time} minutes")
        if inter_time > time_cost:
            print(f"   ⚠️ This route takes {inter_time - time_cost} minutes longer than the fastest route.")

if __name__ == "__main__":
    main()